# 02 原型测试

本笔记本进行离线最小训练原型验证，目标是**快速功能验证**而非最终性能报告。

1. **冒烟测试**：验证 `TrajectoryDataHandler`、`DOADataModule`、模型前向传播、卡尔曼滤波器
2. **最小训练原型**：微型模型 2 epoch 端到端训练验证
3. **快速比较实验**：不同轨迹配置的 RMSPE / 运行时间对比
4. **综合结果汇总**：质量门检查，决定是否进入正式训练

### 笔记本规范

- 笔记本用于探索和验证，可重用逻辑应迁移至 `src/`
- 按编号命名以保持长期可维护性
- 所有冒烟测试返回结构化 `StepResult`，便于自动化集成

## 1) 环境设置

初始化路径、随机种子和输出目录。导入顺序确保本地 `src` 优先于 `DCD_MUSIC.src`。

In [ ]:
from __future__ import annotations

# STEP 01: ????????????????? notebook ??????
# step 1.1: ????????????????????????? code cell ?????????

import copy
import math
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# STEP 02: ?????????????????????
# step 2.1: ??????????? notebooks????????????????????
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# step 2.2: ???????? sys.path??????? config/ ? src/ ?????
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# step 2.3: ? DCD_MUSIC ????????????????????
DCD_ROOT = ROOT / "DCD_MUSIC"
if str(DCD_ROOT) not in sys.path:
    sys.path.append(str(DCD_ROOT))

# step 2.4: ?????????parents=True ???????????
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# STEP 03: ???????????????????????
plt.rcParams.update({
    "figure.dpi": 150,      # ??????? 150??????? notebook ?????
    "axes.grid": True,      # ????????????????????
    "grid.alpha": 0.3,      # ????????????????
    "grid.linestyle": "--",# ????????????
    "font.size": 10,        # ????????????????
})
np.random.seed(42)
torch.manual_seed(42)

# STEP 04: ??????????? DCD_MUSIC ??????????
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dcd_ok = DCD_ROOT.exists() and any(DCD_ROOT.iterdir())

print(f"Project root : {ROOT}")
print(f"Device       : {device}")
print(f"DCD_MUSIC    : {'OK' if dcd_ok else 'MISSING - run: git submodule update --init --recursive'}")
print(f"Figure output: {FIG_DIR}")


## 2) 导入和工具函数

定义：
- `StepResult`：结构化测试结果记录
- `build_fast_config()`：快速配置覆盖（小样本量、短轨迹、少 epoch）
- `format_status_table()`：统一表格报告

In [ ]:
# STEP 05: ?????????????????? notebook ??????
# step 5.1: ??????????????????????????????

from config.loader import load_config
from config.utils import create_system_model
from src.model_module import SubspaceNetLightning
from src.data_module.trajectory import TrajectoryDataHandler
from src.data_module.lit_datamodule import DOADataModule
from src.eval_module.metrics.rmspe_loss import RMSPELoss


@dataclass
class StepResult:
    """??????????

    Attributes:
        name: ???????????????????
        ok: ???????????
        message: ??????????
        elapsed_s: ??????????
    """
    name: str
    ok: bool
    message: str
    elapsed_s: float


def build_fast_config(
    samples_size: int = 48,
    trajectory_length: int = 12,
    batch_size: int = 8,
    epochs: int = 2,
    trajectory_type: str = "random_walk",
    random_walk_std_dev: float = 2.0,
):
    """???????????????

    Args:
        samples_size: ??????????????? notebook ?????
        trajectory_length: ????????????????????????
        batch_size: ??????????????????
        epochs: ????????????????????????
        trajectory_type: ??????? random_walk ? static?
        random_walk_std_dev: ?????????????????

    Returns:
        ?????????????
    """
    cfg = load_config(str(ROOT / "run" / "conf" / "default_config.yaml"))

    # step 5.2: ??????????????????????????
    cfg.dataset.samples_size = samples_size
    cfg.dataset.create_data = True
    cfg.dataset.save_dataset = False

    # step 5.3: ??????????????????????
    cfg.trajectory.enabled = True
    cfg.trajectory.trajectory_length = trajectory_length
    cfg.trajectory.trajectory_type = trajectory_type
    cfg.trajectory.random_walk_std_dev = float(random_walk_std_dev)
    cfg.trajectory.save_trajectory = False

    # step 5.4: ??????????????? notebook ???????????
    cfg.training.batch_size = batch_size
    cfg.training.epochs = epochs
    cfg.training.learning_rate = 1e-3  # 1e-3 ???????? Adam ??????

    # step 5.5: ???????????? batch ??????????? RMSPE ???
    cfg.system_model.M = 3
    return cfg


def format_status_table(rows: list[StepResult]) -> pd.DataFrame:
    """? StepResult ?????? DataFrame?

    Args:
        rows: ??????????

    Returns:
        ?? display() ??????????
    """
    return pd.DataFrame([
        {
            "???": r.name,
            "??": "PASS" if r.ok else "FAIL",
            "??(s)": round(r.elapsed_s, 4),
            "??": r.message,
        }
        for r in rows
    ])


print(f"???????: {device}")


## 3) 冒烟测试 A：TrajectoryDataHandler

验证目标：
- 数据集生成无异常
- 返回的 `TrajectoryDataset` 结构和形状正确
- `__getitem__` 返回 `(X, sources_num, labels)` 三元组

In [ ]:
def smoke_test_trajectory_handler(cfg) -> StepResult:
    """?????TrajectoryDataHandler ??????

    Args:
        cfg: ????????

    Returns:
        StepResult: ???????????????????????
    """
    start = time.perf_counter()
    name = "TrajectoryDataHandler"

    # STEP 06: ???????????????????????????????
    if not dcd_ok:
        return StepResult(name, False, "SKIP: DCD_MUSIC ???????", time.perf_counter() - start)

    try:
        # step 6.1: ?????????????
        system_model = create_system_model(cfg)
        handler = TrajectoryDataHandler(system_model_params=system_model.params, config=cfg)
        ds, _ = handler.create_dataset(
            samples_size=cfg.dataset.samples_size,
            trajectory_length=cfg.trajectory.trajectory_length,
            trajectory_type=cfg.trajectory.trajectory_type,
            save_dataset=False,
        )

        # step 6.2: ?????????????????????????
        x0, m0, y0 = ds[0]

        # step 6.3: ???????? L???? N???? T???????????????????
        expected_L = cfg.trajectory.trajectory_length
        expected_N = cfg.system_model.N
        expected_T = cfg.system_model.T
        assert x0.shape[0] == expected_L, f"L ???: {x0.shape[0]} != {expected_L}"
        assert x0.shape[1] == expected_N, f"N ???: {x0.shape[1]} != {expected_N}"
        assert x0.shape[2] == expected_T, f"T ???: {x0.shape[2]} != {expected_T}"
        assert len(m0) == expected_L, f"sources_num ?????: {len(m0)} != {expected_L}"

        msg = f"len={len(ds)}, X={tuple(x0.shape)} [L,N,T], sources_num_len={len(m0)}, labels_len={len(y0)}"
        return StepResult(name, True, msg, time.perf_counter() - start)
    except Exception as exc:
        return StepResult(name, False, f"{type(exc).__name__}: {exc}", time.perf_counter() - start)


# step 6.4: ????????????????????
cfg_base = build_fast_config()
res_handler = smoke_test_trajectory_handler(cfg_base)
format_status_table([res_handler])


## 4) 冒烟测试 B：DOADataModule

验证目标：
- `setup('fit')` 正确创建 train/val/test 分割
- DataLoader 返回有效批次
- 张量和标签维度对齐

In [ ]:
def smoke_test_datamodule(cfg) -> StepResult:
    """?????DOADataModule ??? DataLoader?

    Args:
        cfg: ????????

    Returns:
        StepResult: ???????? batch ???????
    """
    start = time.perf_counter()
    name = "DOADataModule"

    # STEP 07: DataModule ???????????????????????
    if not dcd_ok:
        return StepResult(name, False, "SKIP: DCD_MUSIC ???????", time.perf_counter() - start)

    try:
        # step 7.1: ?? Lightning DataModule ??? fit ??? setup?
        system_model = create_system_model(cfg)
        dm = DOADataModule(cfg, system_model)
        dm.setup("fit")

        # step 7.2: ?? split ??????????/??????????????????
        assert len(dm.train_dataset) > 0, "?????"
        assert len(dm.val_dataset) > 0, "?????"
        assert len(dm.test_dataset) > 0, "?????"

        # step 7.3: ???????????? batch ?????????
        total = len(dm.train_dataset) + len(dm.val_dataset) + len(dm.test_dataset)

        train_dl = dm.train_dataloader()
        batch = next(iter(train_dl))
        x, m, y = batch

        msg = (
            f"train={len(dm.train_dataset)}, val={len(dm.val_dataset)}, "
            f"test={len(dm.test_dataset)}, total={total}, "
            f"batch: X={tuple(x.shape)}, M={tuple(m.shape)}, Y={tuple(y.shape)}"
        )
        return StepResult(name, True, msg, time.perf_counter() - start)
    except Exception as exc:
        return StepResult(name, False, f"{type(exc).__name__}: {exc}", time.perf_counter() - start)


res_dm = smoke_test_datamodule(cfg_base)
format_status_table([res_dm])


## 5) 冒烟测试 C：模型前向传播

验证目标：
- `SubspaceNetLightning` 实例化成功
- 至少一种前向调用签名正常工作
- 输出格式可用于损失计算

In [ ]:
def smoke_test_model_forward(cfg) -> StepResult:
    """?????SubspaceNetLightning ???????

    Args:
        cfg: ????????

    Returns:
        StepResult: ???????????????????
    """
    start = time.perf_counter()
    name = "SubspaceNet ????"

    # STEP 08: ?????????????????????????????????
    if not dcd_ok:
        return StepResult(name, False, "SKIP: DCD_MUSIC ???????", time.perf_counter() - start)

    try:
        # step 8.1: ? DataModule ??????? batch???????????????
        system_model = create_system_model(cfg)
        dm = DOADataModule(cfg, system_model)
        dm.setup("fit")
        x, m, _ = next(iter(dm.train_dataloader()))

        # step 8.2: ????????????????? CPU/GPU ?????
        model = SubspaceNetLightning(system_model=system_model).to(device)
        x = x.to(device)
        m = m.to(device)

        # step 8.3: ?????????????????????????????
        out = None
        last_err = None
        signatures = [
            ("full trajectory", lambda: model(x, m)),
            ("full no m", lambda: model(x)),
            ("last step + m", lambda: model(x[:, -1], m[:, -1])),
            ("last step only", lambda: model(x[:, -1])),
        ]
        used_sig = ""
        for sig_name, call in signatures:
            try:
                out = call()
                used_sig = sig_name
                break
            except Exception as exc:
                last_err = exc

        if out is None:
            raise RuntimeError(f"?????????; ????: {last_err}")

        # step 8.4: ?????? tensor????? tuple/list???????????????
        if isinstance(out, (tuple, list)):
            out_desc = ", ".join(
                f"{tuple(t.shape)}" for t in out if hasattr(t, "shape")
            )
        elif hasattr(out, "shape"):
            out_desc = str(tuple(out.shape))
        else:
            out_desc = f"type={type(out).__name__}"

        msg = f"??='{used_sig}', output=({out_desc})"
        return StepResult(name, True, msg, time.perf_counter() - start)
    except Exception as exc:
        return StepResult(name, False, f"{type(exc).__name__}: {exc}", time.perf_counter() - start)


res_fwd = smoke_test_model_forward(cfg_base)
format_status_table([res_fwd])


## 6) 冒烟测试 D：卡尔曼滤波器

验证目标：
- `ExtendedKalmanFilter1D` 实例化和 `predict/update` 循环无异常
- 新息统计量（`innovation`、`Ky`、`y_S_inv_y`）返回值合理
- 这是在线无监督自适应的核心组件

In [ ]:
def smoke_test_kalman_filter(cfg) -> StepResult:
    """?????ExtendedKalmanFilter1D ??/?????

    Args:
        cfg: ????????????????? M?

    Returns:
        StepResult: ?? 1D EKF ???????????
    """
    start = time.perf_counter()
    name = "ExtendedKalmanFilter1D"

    try:
        from simulation.kalman_filter import ExtendedKalmanFilter1D

        # STEP 09: ????????? EKF???????????????????
        M = cfg.system_model.M
        kf = ExtendedKalmanFilter1D(
            num_sources=M,
            process_noise=0.01,      # ????????????????????
            measurement_noise=1e-3,  # ???????????????????
            initial_covariance=1.0,  # ??????? 1.0????????????
        )

        # step 9.1: ?????????????????????????????
        init_angles = np.linspace(-20, 20, M)
        kf.initialize(init_angles)

        # step 9.2: ???? 10 ???/????????????????????
        innovations = []
        for step in range(10):
            kf.predict()

            # step 9.3: ???????????????????????
            true_angles = init_angles + step * 0.5 + np.random.randn(M) * 0.1
            result = kf.update(true_angles)

            if hasattr(result, "innovation"):
                innovations.append(float(np.mean(np.abs(result.innovation))))

        msg = (
            f"M={M}, 10???/????"
            + (f", ??|innovation|={np.mean(innovations):.4f}" if innovations else "")
        )
        return StepResult(name, True, msg, time.perf_counter() - start)
    except ImportError:
        return StepResult(name, False, "SKIP: simulation.kalman_filter ????", time.perf_counter() - start)
    except Exception as exc:
        return StepResult(name, False, f"{type(exc).__name__}: {exc}", time.perf_counter() - start)


res_kf = smoke_test_kalman_filter(cfg_base)
format_status_table([res_kf])


## 7) 最小离线训练原型

构建一个微型角度回归模型，经过 2 个 epoch 训练，验证优化器、损失计算和运行时日志记录是否端到端正常工作。

In [ ]:
class TinyAngleRegressor(nn.Module):
    """???????????????????

    ?????????????????????????????????
    """

    def __init__(self, n_sensors: int, n_sources: int):
        """????? MLP ????

        Args:
            n_sensors: ??? N?
            n_sources: ????????? M?
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * n_sensors, 64),  # 2N ????????????
            nn.ReLU(),
            nn.Linear(64, n_sources),
        )

    def forward(self, x_last_step: torch.Tensor) -> torch.Tensor:
        """??????????????????????

        Args:
            x_last_step: ???????????????? [B, N, T]?

        Returns:
            torch.Tensor: ???????? [B, M]?
        """
        # STEP 10: ?????????????????????????
        feat_real = x_last_step.real.mean(dim=-1)  # [B, N, T] -> [B, N]
        feat_imag = x_last_step.imag.mean(dim=-1)  # [B, N, T] -> [B, N]

        # step 10.1: ?????????????? [B, 2N] ????????
        feat = torch.cat([feat_real, feat_imag], dim=-1)
        return self.net(feat)


def run_tiny_prototype(cfg) -> Dict[str, Any]:
    """?????????????? RMSPE?

    Args:
        cfg: ????????

    Returns:
        Dict[str, Any]: ??????????RMSPE?? epoch ????????
    """
    t0 = time.perf_counter()

    # STEP 11: ???????????????????? NaN ???????
    if not dcd_ok:
        return {
            "status": "SKIP", "message": "DCD_MUSIC ???????",
            "loss": np.nan, "rmspe_deg": np.nan, "runtime_s": time.perf_counter() - t0,
        }

    # step 11.1: ??????? DataModule????????????????
    system_model = create_system_model(cfg)
    dm = DOADataModule(cfg, system_model)
    dm.setup("fit")

    train_dl = dm.train_dataloader()
    n_sensors = int(cfg.system_model.N)
    n_sources = int(cfg.system_model.M)

    # step 11.2: ???????Adam ???? RMSPE ?????
    model = TinyAngleRegressor(n_sensors=n_sensors, n_sources=n_sources).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.training.learning_rate)
    criterion = RMSPELoss().to(device)

    epoch_losses_history = []

    # step 11.3: ???? epoch????????????????????
    for epoch in range(cfg.training.epochs):
        model.train()
        epoch_losses = []

        for x, _, y in train_dl:
            x, y = x.to(device), y.to(device)

            # step 11.4: ????????????????????????
            x_last = x[:, -1]                     # [B, L, N, T] -> [B, N, T]
            y_last = y[:, -1, :n_sources]         # [B, L, *] -> [B, M]

            pred = model(x_last)
            loss = criterion(pred, y_last) / max(1, x_last.size(0))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(float(loss.detach().cpu()))

        epoch_losses_history.append(np.mean(epoch_losses) if epoch_losses else np.nan)

    # step 11.5: ????? loss ????????????? DOA ???????
    last_loss = epoch_losses_history[-1] if epoch_losses_history else np.nan
    last_rmspe = float(last_loss * 180.0 / math.pi) if not np.isnan(last_loss) else np.nan

    return {
        "status": "OK",
        "message": "??????",
        "loss": float(last_loss),
        "rmspe_deg": last_rmspe,
        "epoch_losses": epoch_losses_history,
        "runtime_s": time.perf_counter() - t0,
    }


proto_result = run_tiny_prototype(cfg_base)
print(f"??: {proto_result['status']}")
print(f"????: {proto_result['loss']:.6f}")
print(f"?? RMSPE: {proto_result['rmspe_deg']:.2f}?")
print(f"????: {proto_result['runtime_s']:.3f}s")
print(f"Epoch ????: {[f'{v:.4f}' for v in proto_result.get('epoch_losses', [])]}")


## 8) 快速比较实验

通过变更 `trajectory_type` 和 `random_walk_std_dev` 比较不同轨迹配置下的训练效果。

In [ ]:
# STEP 12: ?????????????????????????????
settings = [
    {"trajectory_type": "random_walk", "random_walk_std_dev": 1.0},
    {"trajectory_type": "random_walk", "random_walk_std_dev": 3.0},
    {"trajectory_type": "random_walk", "random_walk_std_dev": 5.0},
    {"trajectory_type": "static",      "random_walk_std_dev": 1.0},
]

rows = []
for s in settings:
    # step 12.1: ?????????????????????????????
    cfg = build_fast_config(
        samples_size=48, trajectory_length=12, batch_size=8, epochs=2,
        trajectory_type=s["trajectory_type"],
        random_walk_std_dev=s["random_walk_std_dev"],
    )
    out = run_tiny_prototype(cfg)
    rows.append({
        "????": s["trajectory_type"],
        "????std": s["random_walk_std_dev"],
        "??": out["status"],
        "??": round(out["loss"], 6) if not np.isnan(out["loss"]) else "N/A",
        "RMSPE [?]": round(out["rmspe_deg"], 2) if not np.isnan(out["rmspe_deg"]) else "N/A",
        "?? [s]": round(out["runtime_s"], 3),
    })

comparison_df = pd.DataFrame(rows)
display(comparison_df)


### 8a) RMSPE 对比柱状图 + 帕累托前沿

左图：不同配置的 RMSPE 对比（越低越好）
右图：运行时间 vs RMSPE 的帕累托前沿，标识最优权衡点

In [ ]:
# STEP 13: ?????????????????????????????

# step 13.1: ??????????????????????????
plot_df = comparison_df.copy()
plot_df["RMSPE_val"] = pd.to_numeric(plot_df["RMSPE [?]"], errors="coerce")
plot_df["runtime_val"] = pd.to_numeric(plot_df["?? [s]"], errors="coerce")
plot_df = plot_df.dropna(subset=["RMSPE_val", "runtime_val"])
plot_df["??"] = plot_df.apply(
    lambda r: f"{r['????']} | std={r['????std']}", axis=1
)
plot_df = plot_df.sort_values("RMSPE_val", ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# step 13.2: ???? RMSPE ????????????????????????
colors = plt.cm.Set2(np.linspace(0, 0.8, len(plot_df)))
bars = ax1.barh(plot_df["??"], plot_df["RMSPE_val"], color=colors, edgecolor="white")
for bar, val in zip(bars, plot_df["RMSPE_val"]):
    ax1.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
             f"{val:.1f}?", va="center", fontsize=9)
ax1.set_xlabel("RMSPE [?] (????)")
ax1.set_title("?? RMSPE ??")

# step 13.3: ?????????????????????????
X = plot_df["runtime_val"].to_numpy()
Y = plot_df["RMSPE_val"].to_numpy()

is_pareto = np.ones(len(plot_df), dtype=bool)
for i in range(len(plot_df)):
    for j in range(len(plot_df)):
        # ??????????????????????????????? Pareto ??
        if i != j and X[j] <= X[i] and Y[j] <= Y[i] and (X[j] < X[i] or Y[j] < Y[i]):
            is_pareto[i] = False
            break

ax2.scatter(X[~is_pareto], Y[~is_pareto], color="#7f7f7f", s=60, label="? Pareto")
ax2.scatter(X[is_pareto], Y[is_pareto], color="#d62728", s=100, zorder=5, label="Pareto ??")

for _, r in plot_df.iterrows():
    ax2.annotate(r["??"], (r["runtime_val"], r["RMSPE_val"]),
                 textcoords="offset points", xytext=(6, 4), fontsize=8)

frontier = plot_df.iloc[np.where(is_pareto)[0]].sort_values("runtime_val")
if len(frontier) >= 2:
    ax2.plot(frontier["runtime_val"], frontier["RMSPE_val"],
             color="#d62728", linewidth=1.5, alpha=0.6, linestyle="--")

ax2.set_xlabel("???? [s] (????)")
ax2.set_ylabel("RMSPE [?] (????)")
ax2.set_title("?????????? vs RMSPE")
ax2.legend(loc="best", fontsize=9)

fig.tight_layout()
fig.savefig(FIG_DIR / "prototype_comparison.png", dpi=180)
plt.show()


## 9) 综合结果汇总

合并所有冒烟测试状态和原型指标，形成单一质量门判断。

In [ ]:
# STEP 14: ???????????????? notebook ???????
all_results = [res_handler, res_dm, res_fwd, res_kf]
smoke_df = format_status_table(all_results)

print("=== ?????? ===")
display(smoke_df)

# step 14.1: ??????????????????????????????
all_pass = all(r.ok for r in all_results)
total_time = sum(r.elapsed_s for r in all_results)

print(f"\n???: {total_time:.3f}s")
if all_pass:
    print("???: PASS - ???????????????????")
else:
    failed = [r.name for r in all_results if not r.ok]
    print(f"???: FAIL - ???????: {', '.join(failed)}")


### 实验日志推荐字段

在正式运行中保持以下字段稳定，便于后续分析对比：

| 字段 | 说明 |
| ---- | ---- |
| `trajectory_type` | 轨迹运动模型 |
| `random_walk_std_dev` | 随机游走标准差 |
| `loss` | 训练损失 |
| `rmspe_deg` | RMSPE（度） |
| `runtime_s` | 运行时间 |
| `status` | 运行状态 |

In [ ]:
# STEP 15: ?????? RMSPE ????????????????
# step 15.1: ???????????????? comparison_df ????????
display(comparison_df.sort_values("RMSPE [?]", ascending=True))


## 10) 导出与下一步

列出导出的图表，提供后续步骤建议。

exported = sorted([p.name for p in FIG_DIR.glob("prototype_*.png")])
print(f"导出 {len(exported)} 张图表到 {FIG_DIR}：")
for name in exported:
    print(f"  ├── {name}")

print(f"\n=== 原型验证完成 ===")
print(f"  冒烟测试: {sum(r.ok for r in all_results)}/{len(all_results)} 通过")
print(f"  原型训练: {proto_result['status']}, RMSPE={proto_result['rmspe_deg']:.1f}°")
print(f"  对比实验: {len(comparison_df)} 组配置已测试")

print("\n后续步骤：")
print("  1. 将冒烟测试迁移至 tests/ 目录（pytest 自动化）")
print("  2. 用 SubspaceNetLightning 替换 TinyAngleRegressor 进行正式训练")
print("  3. 启用在线学习流水线（OnlineLearningPipeline）验证自适应能力")
print("  4. 扩大超参数网格搜索（SNR、eta、轨迹类型）")